In [5]:
import os
import time
import random
import pandas as pd
from selenium import webdriver
from selenium.webdriver.edge.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

edge_options = Options()
edge_options.add_experimental_option("excludeSwitches", ["enable-automation"])
edge_options.add_experimental_option("useAutomationExtension", False)

# Set Page Load Strategy to eager so it doesn't hang on background media
edge_options.page_load_strategy = 'eager'

local_app_data = os.getenv('LOCALAPPDATA')
everyday_profile = os.path.join(local_app_data, r"Microsoft\Edge\User Data")
edge_options.add_argument(f"user-data-dir={everyday_profile}")
edge_options.add_argument("profile-directory=Default")

driver = webdriver.Edge(options=edge_options)
driver.execute_cdp_cmd(
    "Page.addScriptToEvaluateOnNewDocument",
    {"source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"}
)
print("Browser initialized with eager loading strategy!")

Browser initialized with eager loading strategy!


In [6]:
from selenium.common.exceptions import TimeoutException

target_url = "https://x.com/Polymarket/status/2092606015644528757?s=20"

# 1. Bypass driver.get() entirely using instant JavaScript navigation
driver.execute_script("window.location.href = arguments[0];", target_url)

print("URL passed to browser. Waiting for the post to render...")

# 2. Wait up to 20 seconds for the actual tweet elements to appear on the screen
try:
    wait = WebDriverWait(driver, 20)
    wait.until(EC.presence_of_element_located((By.XPATH, "//article[@data-testid='tweet']")))
    print("Target post loaded successfully!")
except TimeoutException:
    print("The page took too long to render the tweets. Check your internet connection or the URL.")

URL passed to browser. Waiting for the post to render...
Target post loaded successfully!


In [7]:
import random
import time
from selenium.webdriver.common.by import By

extracted_data = []
seen_tweet_signatures = set()

# Safety settings
MAX_COMMENTS = 250  # Hard stop limit to prevent continuous scraping
no_new_data_counter = 0
max_empty_scrolls = 4
scroll_cycle = 0

print(f"Starting stealth extraction (Target limit: {MAX_COMMENTS} comments)...")

while no_new_data_counter < max_empty_scrolls and len(extracted_data) < MAX_COMMENTS:
    scroll_cycle += 1
    
    # 1. Capture currently visible tweets
    tweets = driver.find_elements(By.XPATH, "//article[@data-testid='tweet']")
    found_new_comment_this_scroll = False
    
    for tweet in tweets:
        try:
            tweet_fingerprint = tweet.text
            if not tweet_fingerprint or tweet_fingerprint in seen_tweet_signatures:
                continue
            
            seen_tweet_signatures.add(tweet_fingerprint)
            found_new_comment_this_scroll = True
            
            # Extract Username
            try:
                username = tweet.find_element(By.XPATH, ".//div[@data-testid='User-Name']").text.split('\n')[0]
            except:
                username = "Unknown"
                
            # Extract Timestamp
            try:
                timestamp = tweet.find_element(By.TAG_NAME, "time").get_attribute("datetime")
            except:
                timestamp = "Unknown"
                
            # Extract Text
            try:
                text = tweet.find_element(By.XPATH, ".//div[@data-testid='tweetText']").text
            except:
                text = "[Media/GIF Only - No Text]"
            
            extracted_data.append({
                "Username": username,
                "Timestamp": timestamp,
                "Comment": text
            })
            
            # Stop early if target limit is reached
            if len(extracted_data) >= MAX_COMMENTS:
                break
        except Exception:
            continue

    # Update empty-scroll counter
    if not found_new_comment_this_scroll:
        no_new_data_counter += 1
        print(f"No new comments loaded. Retry {no_new_data_counter}/{max_empty_scrolls}...")
    else:
        no_new_data_counter = 0
        print(f"Progress: {len(extracted_data)} comments collected so far...")

    # 2. Smooth incremental scroll (simulates mouse-wheel scrolling)
    total_scroll_distance = random.randint(700, 1100)
    step = 0
    while step < total_scroll_distance:
        increment = random.randint(120, 250)
        driver.execute_script(f"window.scrollBy(0, {increment});")
        step += increment
        time.sleep(random.uniform(0.15, 0.35))
    
    # 3. Base human reading delay
    time.sleep(random.uniform(4.5, 7.5))
    
    # 4. Periodic "deep reading" pause every 10 cycles
    if scroll_cycle % 10 == 0:
        pause_time = random.uniform(10.0, 15.0)
        print(f"--- Simulating reading pause ({pause_time:.1f}s) ---")
        time.sleep(pause_time)

print(f"\nExtraction complete! Safely captured {len(extracted_data)} records.")

Starting stealth extraction (Target limit: 250 comments)...
Progress: 20 comments collected so far...
Progress: 27 comments collected so far...
Progress: 33 comments collected so far...
Progress: 37 comments collected so far...
Progress: 47 comments collected so far...
Progress: 53 comments collected so far...
Progress: 57 comments collected so far...
Progress: 64 comments collected so far...
Progress: 67 comments collected so far...
Progress: 74 comments collected so far...
--- Simulating reading pause (10.3s) ---
Progress: 77 comments collected so far...
Progress: 83 comments collected so far...
Progress: 93 comments collected so far...
Progress: 95 comments collected so far...
Progress: 98 comments collected so far...
Progress: 105 comments collected so far...
Progress: 114 comments collected so far...
Progress: 116 comments collected so far...
Progress: 131 comments collected so far...
Progress: 136 comments collected so far...
--- Simulating reading pause (13.4s) ---
Progress: 144

In [8]:
import pandas as pd

driver.quit()

df = pd.DataFrame(extracted_data)
export_filename = "polymarket_safe_extraction.csv"
df.to_csv(export_filename, index=False, encoding="utf-8")

print(f"Success! {len(df)} records safely saved to {export_filename}.")

Success! 250 records safely saved to polymarket_safe_extraction.csv.


In [9]:
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# 1. Load the data you just extracted
df = pd.read_csv("polymarket_safe_extraction.csv")

# 2. Initialize the VADER analyzer
analyzer = SentimentIntensityAnalyzer()

# 3. Create a function to categorize the text
def get_sentiment(text):
    # Handle any empty rows safely
    if pd.isna(text):
        return "Neutral"
    
    # Calculate the sentiment score
    scores = analyzer.polarity_scores(str(text))
    compound = scores['compound']
    
    # Categorize based on the compound score
    if compound >= 0.05:
        return "Positive"
    elif compound <= -0.05:
        return "Negative"
    else:
        return "Neutral"

# 4. Apply the function to create a new Sentiment column
df['Sentiment'] = df['Comment'].apply(get_sentiment)

# 5. Preview the results and save the enriched data
print(df['Sentiment'].value_counts())
df.to_csv("polymarket_sentiment_scored.csv", index=False, encoding="utf-8")
print("\nSentiment analysis complete! New file saved.")

Sentiment
Negative    104
Neutral      88
Positive     58
Name: count, dtype: int64

Sentiment analysis complete! New file saved.
